# Day 6—Putting a number on a text
*Measuring Manuscripts*

Two classic measurements. **Zipf's law** ranks words by frequency and finds a strikingly regular curve. **Shannon entropy** reduces a text to one number: its average surprise. Both run on the sample. The harder task is deciding whether the number means anything.

## 1. Setup and text

> 🔧 **TO BUILD**—swap in the real material before class. Everything else runs as is.

In [ ]:
import numpy as np, math
import matplotlib.pyplot as plt
from collections import Counter

# === TO BUILD: replace `sample` with a real text (reuse your Day 3 corpus) ===
sample = (
    "Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, and what is the use of a book, thought Alice, without pictures or conversations. So she was considering in her own mind, as well as she could, for the hot day made her feel very sleepy and stupid, whether the pleasure of making a daisy chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her. There was nothing so very remarkable in that nor did Alice think it so very much out of the way to hear the Rabbit say to itself, oh dear, oh dear, I shall be late. But when the Rabbit actually took a watch out of its waistcoat pocket and looked at it and then hurried on, Alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistcoat pocket or a watch to take out of it, and burning with curiosity she ran across the field after it."
)
tokens = [w.lower().strip('.,;:!?') for w in sample.split()]
print(len(tokens), 'tokens,', len(set(tokens)), 'types')

## 2. The Zipf curve

Rank the words from most to least common and plot rank against frequency on log–log axes. A nearly straight line appears—the signature of Zipf's law.

In [ ]:
freq = Counter(tokens)
counts = np.array(sorted(freq.values(), reverse=True))
ranks = np.arange(1, len(counts) + 1)

plt.figure(figsize=(6, 4))
plt.loglog(ranks, counts, marker='o', linestyle='none')
plt.xlabel('rank (log)'); plt.ylabel('frequency (log)')
plt.title('Zipf: rank vs frequency'); plt.grid(alpha=0.3); plt.show()

## 3. Fit the slope

Zipf's law predicts a straight line with slope near −1. Fit a line to the log–log points and read off the slope. A short text is noisy, so don't expect exactly −1.

In [ ]:
slope, intercept = np.polyfit(np.log(ranks), np.log(counts), 1)
print('Fitted slope:', round(slope, 2), '   (Zipf predicts about -1)')

## 4. Entropy: average surprise

High entropy means the next word is hard to predict; low entropy means it's easy. One number for the whole distribution, measured in bits.

In [ ]:
def entropy(items):
    n = len(items)
    return -sum((c / n) * math.log2(c / n) for c in Counter(items).values())

print('Word entropy:', round(entropy(tokens), 2), 'bits per word')

## 5. Characters vs words, and redundancy

Measure entropy at the character level too. Comparing it to the maximum possible entropy—if every symbol were equally likely—tells you how redundant the writing system is.

In [ ]:
chars = [c for c in sample.lower() if c.isalpha()]
h_char = entropy(chars)
h_max = math.log2(len(set(chars)))
print('Character entropy:', round(h_char, 2), 'bits')
print('Maximum possible: ', round(h_max, 2), 'bits (if all letters were equally likely)')
print('Redundancy:       ', round(100 * (1 - h_char / h_max)), '%')

## 6. Compare two texts

Measurement is most useful in comparison. Shuffle the text into random word order and remeasure: entropy barely moves, because it ignores order entirely. That's a real limitation, worth knowing before you rely on it.

In [ ]:
import random
shuffled = tokens[:]; random.Random(0).shuffle(shuffled)
print('Original entropy:', round(entropy(tokens), 3))
print('Shuffled entropy:', round(entropy(shuffled), 3))
print('Same numbers — word-entropy says nothing about word order.')

## 7. The skeptic's test: random text is Zipfian too

This is the warning the day builds to. Generate gibberish—random letters with random spaces, no language at all—and it also produces a clean Zipf curve. A Zipf curve on your data proves little by itself. Always ask what a meaningless version would look like.

In [ ]:
rng = random.Random(1)
alphabet = 'abcdefghijklmnopqrstuvwxyz      '   # letters + spaces
gibberish = ''.join(rng.choice(alphabet) for _ in range(8000)).split()
g = np.array(sorted(Counter(gibberish).values(), reverse=True))

plt.figure(figsize=(6, 4))
plt.loglog(np.arange(1, len(g) + 1), g, marker='.', linestyle='none')
plt.title('Zipf curve of pure random gibberish'); plt.xlabel('rank'); plt.ylabel('frequency')
plt.grid(alpha=0.3); plt.show()
print('Looks just as Zipfian. Pattern-spotting is easy; meaning is the hard part.')

### Check-in
- What single number or distribution could your project measure?
- What would a null version of your data look like? If you can't tell it from your real result, you don't have a result yet.